# Notebook 01 · Tải subset ASL Citizen + EDA (chạy **một lần**)

**Cách dùng:** clone repo → mở notebook này → *Run All*. Mọi bước nặng đều idempotent.

| Bước | Việc | Cần mạng | Chạy lại thì sao |
|---|---|---|---|
| 1 | Preflight, đọc *central directory* của ZIP (~46 GB) bằng HTTP Range, **không tải ZIP** | ✅ | bỏ qua nếu đã có `_SUBSET_READY.json` |
| 2 | Tải metadata + video của `pool_glosses` gloss ứng viên (ước ≈1.7 GB) | ✅ | chỉ tải phần thiếu/lỗi |
| 3 | Thống kê video + probe MediaPipe (tỉ lệ thấy tay, mức chuyển động) | – | chỉ tính phần thiếu |
| 4 | EDA metadata (không cần video) và EDA video | – | rẻ, chạy lại thoải mái |
| 5 | Chọn tier **core5 ⊂ core20 ⊂ core50**, contact sheet, ghi `train/val/test.csv` | – | rẻ, sửa `OVERRIDE` rồi chạy lại từ đó |

**Trước khi chạy trên Kaggle:** Settings → *Internet = ON* (cần xác minh số điện thoại).
**Sau lần chạy đầu:** *Save Version* → *New Dataset* từ thư mục `asl_outputs` → attach dataset đó vào notebook 02/03. Từ đó không bao giờ phải tải lại.

## 1. Setup

In [ ]:
import sys, importlib, subprocess, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

def _find_root():
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / "configs" / "config.yaml").exists():
            return d
    raise FileNotFoundError("Chạy notebook từ bên trong repo đã clone (thiếu configs/config.yaml).")

ROOT = _find_root()
sys.path.insert(0, str(ROOT))
for pkg in ("remotezip", "tqdm"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from src.data import asl_citizen as A
print("repo:", ROOT)

## 2. Cấu hình (sửa ở đây, không sửa chỗ khác)
- `pool_glosses`: số gloss ứng viên tải về một lần. Là *superset* cho mọi tier → mở rộng class sau này **không phải tải thêm**.
- Ngưỡng `min_*`: số video tối thiểu mỗi gloss ở train/val/test (split chính thức là signer-independent).
- `max_download_gb`: chặn cứng; notebook in dung lượng thật trước khi tải.

In [ ]:
CFG = dict(
    pool_glosses=100,
    min_train=10, min_val=3, min_test=8,
    max_download_gb=8,
    run_probe=True, probe_frames=16,        # MediaPipe probe (tắt: run_probe=False)
    tier_sizes=(5, 20, 50), default_tier="core5", min_det_rate=0.85,
    seed=42,
)
P = A.resolve_paths(ROOT)
print("Kaggle:", P.kaggle, "| đọc dữ liệu từ:", P.src, "(Kaggle Dataset input)" if P.from_input else "")
print("videos  →", P.videos, "\nsplits  →", P.splits)

## 3. Tải subset (bước duy nhất tốn thời gian & mạng)
Lỗi mạng giữa chừng? Chỉ cần chạy lại cell này: file đã tải đủ size được bỏ qua, ghi file theo kiểu atomic (`.part` → rename), CRC-32 được kiểm khi đọc.

In [ ]:
pool = A.build_subset(CFG, P)
pool.head(3)

## 4. EDA A — metadata chính thức (không cần video)

In [ ]:
meta = pd.read_csv(P.manifests / "meta_all.csv")
summary = (meta.groupby("split").agg(videos=("video_id", "size"), glosses=("gloss", "nunique"),
                                      signers=("signer_id", "nunique")).reindex(["train", "val", "test"]))
summary.loc["ALL"] = [len(meta), meta.gloss.nunique(), meta.signer_id.nunique()]
display(summary)

ov = A.signer_overlap(meta)
print("Signer trùng giữa các split:", {k: len(v) for k, v in ov.items()}, "→ mong đợi toàn 0 (signer-independent)")
print("video_id trùng:", int(meta.video_id.duplicated().sum()), "| không khớp file trong zip:", int(meta.zip_name.isna().sum()))

counts = A.gloss_split_counts(meta)
combos = [(10, 3, 8), (8, 2, 6), (12, 4, 10), (15, 4, 12)]
elig = pd.DataFrame([{"min_train/val/test": f"{a}/{b}/{c}",
                      "gloss đạt": int(((counts.train >= a) & (counts.val >= b) & (counts.test >= c)).sum())}
                     for a, b, c in combos])
display(elig)

multi = meta.gloss.str.contains(r"[_\s]").mean()
print(f"Gloss nhiều từ/có gạch dưới: {multi:.1%} | ví dụ:", meta.gloss[meta.gloss.str.contains(r"[_\s]")].drop_duplicates().head(5).tolist())

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
counts.total.plot.hist(bins=30, ax=ax[0]); ax[0].set_title("Số video / gloss")
meta.groupby("signer_id").size().sort_values(ascending=False).plot.bar(ax=ax[1], width=.9)
ax[1].set_title("Số video / signer (long-tail?)"); ax[1].set_xticks([])
counts[["train", "val", "test"]].plot.box(ax=ax[2]); ax[2].set_title("Video / gloss theo split")
plt.tight_layout(); fig.savefig(P.plots / "eda_meta.png", dpi=130); plt.show()
summary.to_csv(P.metrics / "eda_meta_summary.csv")

## 5. EDA B — video của pool (kích thước, fps, độ dài, lỗi)

In [ ]:
pool = pd.read_csv(P.manifests / "manifest_pool.csv")
bad = pool[~pool.ok.astype(bool)]
print(f"Video lỗi/không đọc được: {len(bad)}/{len(pool)}")
display(pool[["fps", "width", "height", "duration", "n_frames"]].describe(percentiles=[.05, .5, .95]).T.round(2))
res = pool.groupby(["width", "height"]).size().sort_values(ascending=False).head(6).rename("videos")
display(res.to_frame())

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
pool.duration.plot.hist(bins=30, ax=ax[0]); ax[0].set_title("Độ dài video (s)")
pool.fps.round().value_counts().sort_index().plot.bar(ax=ax[1]); ax[1].set_title("FPS")
pool.det_rate.dropna().plot.hist(bins=20, ax=ax[2]) if pool.det_rate.notna().any() else ax[2].text(.1, .5, "không có probe MediaPipe")
ax[2].set_title("Tỉ lệ frame thấy tay (probe, mỗi video)")
plt.tight_layout(); fig.savefig(P.plots / "eda_video.png", dpi=130); plt.show()

d = pool[pool.ok.astype(bool)].duration
print(f"Độ dài: p50={d.quantile(.5):.2f}s  p90={d.quantile(.9):.2f}s  p95={d.quantile(.95):.2f}s "
      f"→ dùng cho quyết định T (sequence_length) ở notebook 02; sau khi trim đoạn tay nghỉ sẽ ngắn hơn.")

## 6. Chọn tier class
`core5` = các gloss **ít chuyển động nhất** trong nhóm MediaPipe thấy tay tốt (`min_det_rate`); `core20`, `core50` mở rộng ngẫu nhiên có seed (để phản ánh độ khó thật). Các tier **lồng nhau**.
Xem contact sheet bên dưới; muốn tự chọn gloss thì điền `OVERRIDE` rồi chạy lại từ cell này (không tải lại gì).

In [ ]:
OVERRIDE = {}        # ví dụ: {"core5": ["BOOK", "WATER", ...]}  — core20/core50 sẽ tự mở rộng từ đó
gq = A.gloss_quality(pool)
display(gq.round(3).head(12))
TIERS = A.select_tiers(gq, CFG["tier_sizes"], CFG["min_det_rate"], CFG["seed"], OVERRIDE)
DEFAULT = CFG["default_tier"]
for k, v in TIERS.items():
    print(f"{k}: {len(v)} class")
print(f"{DEFAULT} =", TIERS[DEFAULT])

In [ ]:
fig = A.contact_sheet(pool, TIERS[DEFAULT], P.videos, per_gloss=3)
fig.savefig(P.plots / f"contact_sheet_{DEFAULT}.png", dpi=110); plt.show()

## 7. Ghi split + kiểm tra rò rỉ

In [ ]:
report = A.write_splits(pool, TIERS, P.splits, default_tier=DEFAULT)
display(pd.DataFrame(report).T)

tr, va, te = (pd.read_csv(P.splits / f"{s}.csv") for s in ("train", "val", "test"))
assert set(tr.signer_id).isdisjoint(va.signer_id) and set(tr.signer_id).isdisjoint(te.signer_id) \
       and set(va.signer_id).isdisjoint(te.signer_id), "Rò rỉ signer!"
assert all((P.videos / f).exists() for f in pd.concat([tr, va, te]).video_file), "Thiếu file video"
assert set(tr.gloss) == set(va.gloss) == set(te.gloss), "Class không đồng đều giữa các split"
print("✅ Không rò rỉ signer | mọi video tồn tại | mọi class có mặt ở cả 3 split")

bal = pd.concat([tr, va, te]).groupby(["gloss", "split"]).size().unstack(fill_value=0)[["train", "val", "test"]]
display(bal)
bal.plot.bar(stacked=True, figsize=(8, 3.5), title=f"Phân bố {DEFAULT} theo split")
plt.tight_layout(); plt.savefig(P.plots / f"balance_{DEFAULT}.png", dpi=130); plt.show()
sd = pool[pool.gloss.isin(TIERS[DEFAULT]) & pool.ok.astype(bool)].duration
print(f"{DEFAULT}: độ dài video p50={sd.median():.2f}s, p90={sd.quantile(.9):.2f}s | signer: "
      f"train={tr.signer_id.nunique()}, val={va.signer_id.nunique()}, test={te.signer_id.nunique()}")

## 8. Chốt & bàn giao
Ghi `dataset_report.json`. Sau đó (chỉ làm một lần):
1. **Save Version → Save & Run All (Commit)**.
2. Trang output của version → **New Dataset** → chọn thư mục `asl_outputs` (gồm `raw/`, `manifests/`, `_SUBSET_READY.json`). Kiểm tra license ASL Citizen trước khi chia sẻ; nếu chưa chắc, để dataset **private** trong nhóm.
3. Attach dataset đó vào notebook 02/03. Cell *Tải subset* sẽ tự nhận `_SUBSET_READY.json` và bỏ qua mạng.
4. Commit `data/splits/*.csv` và `data/splits/tiers/` vào Git (nhỏ, cần cho cả nhóm).

In [ ]:
rep = dict(n_pool_videos=int(len(pool)), n_pool_glosses=int(pool.gloss.nunique()),
           bad_videos=int((~pool.ok.astype(bool)).sum()), tiers={k: v for k, v in TIERS.items()},
           default_tier=DEFAULT, splits=report, cfg={k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()})
(P.metrics / "dataset_report.json").write_text(json.dumps(rep, indent=2, default=str))
print("Đã ghi", P.metrics / "dataset_report.json")